In [ ]:
import os
import gc
from dataclasses import dataclass
from typing import List, Dict, Union
import torchaudio.functional as F_audio
from sklearn.utils.class_weight import compute_class_weight
from tqdm.notebook import tqdm
import librosa
import numpy as np
import torch
from datasets import load_dataset, Dataset
from transformers import AutoFeatureExtractor
from sklearn.model_selection import StratifiedGroupKFold

visec_dataset = load_dataset("hustep-lab/ViSEC", split="train")

def extract_samples_generator():
    anon_counter = 0
    for sample in tqdm(visec_dataset, desc="Data Extraction"):
        audio_data = sample["path"]
        arr = np.array(audio_data["array"], dtype=np.float32).ravel()
        sr = audio_data["sampling_rate"]

        arr_trimmed, _ = librosa.effects.trim(arr, top_db=60)
        if len(arr_trimmed) > 0:
            arr = arr_trimmed

        duration = len(arr) / sr
        if duration < 0.5:
          continue
        label_val = LABEL2ID[sample["emotion"]] if isinstance(sample["emotion"], str) else int(sample["emotion"])

        raw_spk = sample["speaker_id"]
        if raw_spk == 0 or str(raw_spk) == "0":
            spk_id = f"anon_{anon_counter}"
            anon_counter += 1
        else:
            spk_id = str(raw_spk)

        if duration <= 12.0:
            yield {"input_array": arr, "sr": sr, "speaker_id": spk_id, "labels": label_val}
        else:
            chunk_length = int(4.0 * sr)
            for i in range(0, len(arr), chunk_length):
                sub_arr = arr[i : i + chunk_length]
                if len(sub_arr) / sr >= 0.5:
                    yield {"input_array": sub_arr, "sr": sr, "speaker_id": spk_id, "labels": label_val}

cleaned_ds = Dataset.from_generator(extract_samples_generator)
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME, do_normalize=False)

def preprocess_dsp(batch: dict) -> dict:
    arr = np.array(batch["input_array"], dtype=np.float32)
    sr = batch["sr"]
    if sr != 16000:
        arr_tensor = torch.from_numpy(arr)
        arr_tensor = F_audio.resample(arr_tensor, orig_freq=sr, new_freq=16000)
        arr = arr_tensor.numpy()
    batch["input_values"] = arr
    return batch

max_cores = os.cpu_count() or 2
processed_ds = cleaned_ds.map(preprocess_dsp, remove_columns=["input_array", "sr"], num_proc=max_cores)

del visec_dataset, cleaned_ds
gc.collect()

@dataclass
class AudioDataCollator:
    feature_extractor: AutoFeatureExtractor
    augment: bool = False

    def __call__(self, features: List[Dict[str, Union[List[float], torch.Tensor, np.ndarray]]]) -> Dict[str, torch.Tensor]:
        augmented = []
        labels = []

        for f in features:
            arr = np.asarray(f["input_values"], dtype=np.float32).copy()
            arr = (arr - np.mean(arr)) / np.sqrt(np.var(arr) + 1e-7)

            if self.augment:
                if np.random.rand() < 0.50:
                    noise_amp = np.random.uniform(0.01, 0.08)
                    noise = np.random.normal(0, noise_amp, arr.shape).astype(np.float32)
                    arr += noise

                if np.random.rand() < 0.50:
                    mask_len = int(len(arr) * np.random.uniform(0.05, 0.15))
                    if mask_len > 0 and len(arr) > mask_len:
                        mask_start = np.random.randint(0, len(arr) - mask_len)
                        arr[mask_start : mask_start + mask_len] = 0.0

            augmented.append({"input_values": arr})
            labels.append(f["labels"])

        batch = self.feature_extractor.pad(
            augmented, padding=True, pad_to_multiple_of=8,
            return_tensors="pt", return_attention_mask=True
        )
        batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch

train_ds = processed_ds
train_meta = processed_ds
all_speakers = np.array(processed_ds["speaker_id"])
all_labels = np.array(processed_ds["labels"])

X_dummy = np.zeros(len(all_labels))
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
cv_folds = []

for train_idx, val_idx in sgkf.split(X_dummy, all_labels, all_speakers):
    cv_folds.append((train_idx, val_idx))

beta = 0.999
classes, counts = np.unique(all_labels, return_counts=True)
effective_num = 1.0 - np.power(beta, counts)
weights = (1.0 - beta) / np.array(effective_num)

weights = weights / np.sum(weights) * len(classes)
alpha_weights = torch.tensor(weights, dtype=torch.float32)

print("\n[KIỂM ĐỊNH PHÂN PHỐI NHÃN TỪNG FOLD]")
for fold_idx, (tr_idx, va_idx) in enumerate(cv_folds, 1):
    val_labels = all_labels[va_idx]
    unique_classes, counts = np.unique(val_labels, return_counts=True)
    class_dist = dict(zip(unique_classes, counts))
    print(f"Fold {fold_idx} Validation - Tổng số mẫu: {len(va_idx)} | Phân phối nhãn (ID: Số lượng): {class_dist}")